# 05 Evaluation And Calibration

Score the trained logistic regression on the held-out test split and export evaluation metrics, predictions, and a calibration plot.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score


def resolve_artifacts_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd / "artifacts", cwd.parent / "artifacts"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return cwd.parent / "artifacts" if cwd.name == "notebooks" else cwd / "artifacts"


ARTIFACTS_DIR = resolve_artifacts_dir()
PLOTS_DIR = ARTIFACTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
model = joblib.load(ARTIFACTS_DIR / "logistic_regression_model.joblib")
X_test = pd.read_csv(ARTIFACTS_DIR / "test_features.csv")
y_test = pd.read_csv(ARTIFACTS_DIR / "test_labels.csv")["blue_team_win"].astype(int)
test_frame = X_test.drop(columns=["date", "blue_team", "red_team"], errors="ignore")
test_prob = model.predict_proba(test_frame)[:, 1]

In [ ]:
metrics = {
    "log_loss": float(log_loss(y_test, test_prob)),
    "brier_score": float(brier_score_loss(y_test, test_prob)),
    "roc_auc": float(roc_auc_score(y_test, test_prob)),
    "accuracy_50": float(accuracy_score(y_test, (test_prob >= 0.5).astype(int))),
}
metrics

In [ ]:
(ARTIFACTS_DIR / "evaluation_metrics.json").write_text(json.dumps(metrics, indent=2))

predictions = pd.DataFrame(
    {
        "blue_win_prob": test_prob,
        "blue_team_win": pd.Series(y_test).astype(int).reset_index(drop=True),
    }
)
predictions.to_csv(ARTIFACTS_DIR / "test_predictions.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_predictions(y_test, test_prob, n_bins=10, ax=ax)
fig.savefig(PLOTS_DIR / "calibration_curve.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
predictions.head()